# MindFlow Step 2b · 5-minute charts, direction, pretrained models

**Question.** Do pretrained models (frozen vision encoders, time-series foundation models) find more than the models trained from scratch in Step 2a? Target: direction (`dir`, `xsdir`).


**A. Frozen vision encoders.** DINOv2-S, CLIP-B/16 and a random ViT-S (floor) read every 234×234 chart resized to
224×224; features [CLS, mean patch] → linear or 2-layer MLP probe, with and without SPY's chart features. Probe
regularisation and early stopping use every 4th validation day, as the Step 2a networks did.

**B. Time-series foundation models**, zero-shot: Chronos-2, TimesFM, Moirai 2 and Kronos forecast the next h
five-minute closes from the last 78 bars (Kronos also reads OHLCV and timestamps). Score:
median forecast return (`dir`), minus its cross-sectional median (`xsdir`).
They run on every `tsfm_every_kth_day`-th validation day (sampling paths is slow).

**Identical rows.** Step 2a's scores (image CNN, ViT, 1D CNN, HGB) are joined on (stock, decision bar); only rows
present in both are kept. Inputs here span one session, and sessions next to a split boundary are removed (embargo).


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing), `Step6d_Pure_Image` (the numbers drawn into the picture, no numbers input) |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing`, `Step7b_Distillation` (small students taught by the final CNN) |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

### Install the foundation-model packages (once per runtime)

In [ ]:
import os, subprocess, sys
if os.environ.get("MINDFLOW_LOCAL_TEST") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting>=2.0", "timesfm[torch]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "uni2ts"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gluonts", "lightning", "einops", "jaxtyping",
                    "hydra-core"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    if not os.path.isdir("/content/Kronos"):
        subprocess.run(["git", "clone", "-q", "https://github.com/shiyu-coder/Kronos", "/content/Kronos"], check=False)

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import chartcnn
import frozen_probe as fp, pretrained_ts as pt
assert chartcnn.__version__ >= "2026-11-23.1", f"stale chartcnn {chartcnn.__version__} in {CODE}"
fp.REFERENCE_MODELS.update({f"tsfm_{k}": pt.label(k) for k in pt.MODELS})
print("frozen_probe", fp.__version__, "| pretrained_ts", pt.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "")

## 2. Configuration
**To resume**, set `RUN_ID`. `REFERENCE_RUN = None` uses the newest Step 2a run; keep `HORIZONS` and `KINDS`
equal to that run's cells. `QUICK = True`: 40,000 training windows, every 5th validation day, DINOv2 + random only,
Chronos-2 only.

**Time on an L4 (estimate):** the full run draws ~300,000 training and ~550,000 validation charts once and feeds each to
every encoder: 20–40 min for DINOv2-S + random ViT-S, 30–60 min more for CLIP-B/16; probes 1–3 min per cell. Chronos-2
and TimesFM: minutes. Kronos: roughly 30–60 min per horizon on every 10th validation day.
Features stay on the Colab disk (`FEATURES_ON_DRIVE = True` also keeps them on Drive, ~5 GB, for resuming).

In [ ]:
SMOKE = LOCAL_TEST
assert DEVICE.type == "cuda" or SMOKE, "Select a GPU runtime"
RUN_ID = None
REFERENCE_RUN = None                               # None = newest Step 2a run
QUICK = False
ENCODER_NAMES = ["dinov2_s", "dinov2_s_random", "clip_b16"]   # add "dinov2_b" for DINOv2 ViT-B/14
PROBES = ["linear", "mlp"]
TSFM = ["randomwalk", "chronos2", "timesfm", "moirai2", "kronos"]   # randomwalk = yardstick, not pretrained
FEATURES_ON_DRIVE = False
SETTINGS = dict(
    stride=12, embargo_sessions=2, max_train_windows=300_000, validation_every_kth_day=1, early_stop_every_kth_day=4,
    extract_batch=256, chunk=50_000, amp=True, num_workers=0 if SMOKE else min(12, os.cpu_count() or 2),
    probe_lr=1e-3, probe_batch=4096, probe_epochs=30, probe_patience=3, linear_l2_grid=[1e-5, 1e-4, 1e-3],
    mlp_hidden=512, mlp_dropout=0.2, mlp_weight_decay_grid=[1e-2, 1e-1], seed=69,
    tsfm_context=78, tsfm_every_kth_day=5, kronos_every_kth_day=10,
)
pt.MODELS["kronos"]["repo"] = "/content/Kronos"
HORIZONS = [12, 1]
KINDS = ["xsdir", "dir"]
if QUICK:
    SETTINGS.update(max_train_windows=40_000, validation_every_kth_day=5, tsfm_every_kth_day=2)
    ENCODER_NAMES, TSFM = ["dinov2_s", "dinov2_s_random"], ["randomwalk", "chronos2"]
if SMOKE:
    SETTINGS.update(max_train_windows=300, extract_batch=16, chunk=128, probe_epochs=3, probe_batch=64,
                    embargo_sessions=1, early_stop_every_kth_day=1, linear_l2_grid=[1e-4], mlp_weight_decay_grid=[1e-2],
                    tsfm_every_kth_day=1)
    ENCODER_NAMES, TSFM = ["dinov2_s", "dinov2_s_random"], ["randomwalk"]
CELLS = [dict(name=f"{k}_h{h}", kind=k, target=fp.KINDS[k], horizon=h, minutes=5 * h, stride=SETTINGS["stride"])
         for h in HORIZONS for k in KINDS]
RUN_ID, RUN_DRIVE, RESUME = mf.new_run(HERE, "2b", RUN_ID, "probe", LOCAL_TEST)
RUN_LOCAL = LOCAL_ROOT / "work" / RUN_DRIVE.parent.name / RUN_ID
RUN_LOCAL.mkdir(parents=True, exist_ok=True)
REF_DIR = mf.latest_run(HERE, "2a", REFERENCE_RUN)
mf.save_settings(RUN_DRIVE, RESUME, settings=SETTINGS, encoders=ENCODER_NAMES, probes=PROBES, tsfm=TSFM, cells=CELLS,
                 quick=QUICK, reference=str(REF_DIR), chartcnn=chartcnn.__version__, frozen_probe=fp.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DRIVE)
print("Step 2a reference run:", REF_DIR if REF_DIR else "none found: pretrained models are compared with each other only")
print(len(CELLS), "cells:", ", ".join(c["name"] for c in CELLS))

## 3. Windows, purge and embargo
One fixed set of windows per split, shared by every cell, encoder, probe and forecaster (saved on Drive). The checks fail
loudly if a validation input reaches into the training period or a training label ends after it.

In [ ]:
store = chartcnn.Store(PREPARED)
cfg = store.config
EMBARGO = fp.embargo_dates(store, SETTINGS["embargo_sessions"])
W = {}
for split, kw in (("train", dict(max_rows=SETTINGS["max_train_windows"])),
                  ("valid", dict(every_kth_day=SETTINGS["validation_every_kth_day"]))):
    path = RUN_DRIVE / f"windows_{split}.parquet"
    if path.exists():
        W[split] = pd.read_parquet(path)
    else:
        W[split] = fp.decision_windows(store, "train" if split == "train" else "validation", SETTINGS["stride"],
                                       EMBARGO, seed=SETTINGS["seed"], **kw)
        W[split].to_parquet(path, index=False)
first_valid_input = store.dates[W["valid"].end_index.min() - fp.LOOKBACK + 1]
last_train_label = store.dates[W["train"].end_index.max() + max(HORIZONS)]
assert first_valid_input > cfg["train_end"], "a validation input window reaches into the training period"
assert last_train_label <= cfg["train_end"], "a training label ends after the training period"
print(f"train windows {len(W['train']):,}, last training label bar on {last_train_label}")
print(f"validation windows {len(W['valid']):,} on {W['valid'].decision_date.nunique()} days, first input bar {first_valid_input}")
SLOTS = np.unique(np.concatenate([W["train"].end_index, W["valid"].end_index]))
SLOT_POS = np.full(store.T, -1, np.int64); SLOT_POS[SLOTS] = np.arange(len(SLOTS))

## 4. A · Frozen encoders: features (parameter counts printed; all frozen)

In [ ]:
ENCODERS = fp.build_encoders(ENCODER_NAMES, DEVICE, tiny=SMOKE, seed=SETTINGS["seed"])
ENCODER_PARAMS = {e: enc.parameters_count()[0] for e, enc in ENCODERS.items()}
FEAT_LOCAL, FEAT_DRIVE = LOCAL_ROOT / f"features_{RUN_DRIVE.parent.name}_{RUN_ID}", RUN_DRIVE / "features"
FEATS = {e: {} for e in ENCODER_NAMES}
kw = dict(batch_size=SETTINGS["extract_batch"], workers=SETTINGS["num_workers"], chunk=SETTINGS["chunk"],
          amp=SETTINGS["amp"], drive_dir=FEAT_DRIVE if FEATURES_ON_DRIVE else None)
t0 = time.perf_counter()
for split in ("train", "valid"):
    out = fp.extract(fp.StockCharts(PREPARED, W[split]), ENCODERS, FEAT_LOCAL, split, DEVICE, **kw)
    for e in out: FEATS[e][split] = out[e]
SPY = fp.extract(fp.SpyCharts(PREPARED, SLOTS), ENCODERS, FEAT_LOCAL, "spy", DEVICE, **kw)
print(f"features ready in {(time.perf_counter() - t0) / 60:.1f} min")
del ENCODERS
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 5. A · Probes per cell

In [ ]:
RESULTS, MEDIANS = {}, {}
for i, cell in enumerate(CELLS, 1):
    print("\n" + "=" * 20, f"[{i}/{len(CELLS)}]", cell["name"], "=" * 20)
    tr, median = fp.cell_rows(store, cell, "train", W["train"])
    va, _ = fp.cell_rows(store, cell, "validation", W["valid"], median)
    MEDIANS[cell["name"]] = median
    assert (store.dates[tr.end_index + cell["horizon"]] <= cfg["train_end"]).all()          # purge
    assert (store.dates[va.end_index + cell["horizon"]] <= cfg["validation_end"]).all()
    data = dict(train=tr, valid=va, feats=FEATS, spy=SPY, slot_pos=SLOT_POS)
    RESULTS[cell["name"]] = fp.run_cell(cell, data, ENCODER_NAMES, PROBES, SETTINGS, DEVICE,
                                        RUN_LOCAL / cell["name"], RUN_DRIVE / cell["name"], smoke=SMOKE)
    if median is not None:
        median.to_csv(RUN_DRIVE / cell["name"] / "train_abs_return_median.csv")

## 6. A · Compare with Step 2a on identical rows
AUC with 95% intervals from resampling whole validation days; paired comparisons use the same days for both models.

In [ ]:
JOINED = {}
for name, r in RESULTS.items():
    pred, share = fp.join_reference(r["predictions"], REF_DIR / name) if REF_DIR else (r["predictions"], None)
    print(f"{name}: {len(pred):,} rows" + ("" if share is None else f", {share:.1%} matched in the reference run"))
    JOINED[name] = dict(info=r["info"], predictions=pred)
TABLE, BOOTS = fp.table(JOINED, n_boot=50 if SMOKE else 300)
TABLE.to_csv(RUN_DRIVE / "probe_table.csv")
MAIN = ["DINOv2-S linear probe", "DINOv2-S MLP probe", "random ViT-S MLP probe", "CLIP-B/16 MLP probe",
        "image CNN (scratch)", "image ViT (scratch)", "HGB hand + SPY"]
for m in MAIN + ["DINOv2-S + SPY MLP probe", "image CNN + SPY panel (scratch)", "1D CNN (scratch)"]:
    if m in set(TABLE.index.get_level_values("model")):
        print(m); display(fp.auc_matrix(TABLE, m))
fp.plot(TABLE, MAIN, RUN_LOCAL / "probe_auc_by_horizon.png")
TAKEAWAYS = fp.takeaways(TABLE, BOOTS)
(RUN_DRIVE / "takeaways.md").write_text(TAKEAWAYS + "\n"); display(Markdown(TAKEAWAYS))
shutil.copy2(RUN_LOCAL / "probe_auc_by_horizon.png", RUN_DRIVE)

## 7. B · Time-series foundation models (zero-shot)
Forecasts on every `tsfm_every_kth_day`-th validation day (Kronos: every `kronos_every_kth_day`-th), cached on Drive.
Each model prints its parameter count. The table compares every model on the rows the foundation model forecast.

In [ ]:
days = np.sort(W["valid"].decision_date.unique())
TS_ROWS, TS_Q, TS_PARAMS = {}, {}, {}
for name in TSFM:
    k = SETTINGS["kronos_every_kth_day"] if name == "kronos" else SETTINGS["tsfm_every_kth_day"]
    keys = W["valid"][W["valid"].decision_date.isin(days[::k])][["security_index", "end_index", "decision_date"]]
    keys = keys.reset_index(drop=True)
    try:
        q, p = pt.forecast_all([name], lambda h, keys=keys: pt.intraday_contexts(PREPARED, store, keys, SETTINGS["tsfm_context"], h),
                               keys, HORIZONS, RUN_DRIVE / "tsfm" / name, DEVICE)
    except Exception as err:
        print(f"{name} skipped: {type(err).__name__}: {err}"); continue
    TS_ROWS[name], TS_Q[name] = keys, q[name]; TS_PARAMS.update(p)
_ = (RUN_DRIVE / "parameters.json").write_text(json.dumps({**{f"frozen_{k}": v for k, v in ENCODER_PARAMS.items()}, **TS_PARAMS}, indent=2))
TS_TABLES = []
for name in TS_ROWS:
    TS_RESULTS = {}
    for cell in CELLS:
        pred = pt.attach(JOINED[cell["name"]]["predictions"], TS_ROWS[name], {name: TS_Q[name][cell["horizon"]]},
                         cell["kind"], ["security_index", "end_index"], "end_index", MEDIANS.get(cell["name"]))
        TS_RESULTS[cell["name"]] = dict(info=JOINED[cell["name"]]["info"], predictions=pred)
    t, _ = fp.table(TS_RESULTS, n_boot=50 if SMOKE else 300)
    t = t.assign(rows_of=pt.label(name))
    TS_TABLES.append(t)
if TS_TABLES:
    TS_TABLE = pd.concat(TS_TABLES)
    TS_TABLE.to_csv(RUN_DRIVE / "tsfm_table.csv")
    keep = ["tsfm_" + n for n in TS_ROWS] + ["dinov2_s_mlp", "cnn_plain", "cnn_vit", "hgb_hand_spy"]
    show = TS_TABLE[TS_TABLE.raw.isin(keep)].reset_index()
    pd.set_option("display.width", 200)
    display(show[["rows_of", "cell", "model", "roc_auc", "auc_ci_low", "auc_ci_high", "rows", "days"]].round(4))

## 8. Parameter counts (advisor rule)

In [ ]:
rows = [dict(model=f"{fp.ENCODERS[e]['label']} (frozen encoder)", parameters=n) for e, n in ENCODER_PARAMS.items()]
rows += [dict(model=pt.label(k[5:]), parameters=v) for k, v in TS_PARAMS.items()]
first = next(iter(RESULTS.values()))["info"]
rows += [dict(model=fp.pretty(k) + " (trainable head)", parameters=v["parameters"]) for k, v in first["probes"].items()]
if REF_DIR and (REF_DIR / "network_parameters.csv").exists():
    ref = pd.read_csv(REF_DIR / "network_parameters.csv")
    rows += [dict(model=f"{r.model} (Step reference, trained)", parameters=r.parameters) for r in ref.itertuples()]
PARAMS = pd.DataFrame(rows).set_index("model"); PARAMS.to_csv(RUN_DRIVE / "parameter_table.csv"); display(PARAMS)

### How to read it
- **Every pretrained model ≈ 0.50**: general visual or time-series pretraining does not create a direction signal.
- **Pretrained probe > random ViT-S**: pretraining changes the features; if both are ≈ 0.5 it does not matter here.